# Revision Re-Analysis Notebook (CF-XHPO / FGCS revision)

**Purpose.** Regenerates, from *existing cached artifacts only* (no GPU retraining), every
quantity needed for the referee-driven revision:

| Section | Reviewer item | Output |
|---|---|---|
| S1 | R2-1: decompose surrogate→model gap (val vs test) | `decomposition_<ds>.csv/tex` + fig |
| S2 | R2-1: AUROC (ranking) vs threshold (F1) as two mechanisms | `two_mechanisms_mvtec.csv` |
| S3 | R1-m1/R2-min: variability (SD/min/max) for Table 4 | `table4_variability_<ds>.csv` |
| S4 | R2-min: MVTec surrogate-vs-confirmed scatter | fig |
| S5 | R2-8: multivariate check of the "corruption driver" claim | `soft_control_regression_<ds>.csv` |
| S6 | R2-min: duplication rate of generated CFs vs logged trials | `duplication_rate_<ds>.csv` |
| S7-S8 | R2-2/3: query suite + retrieval baselines, raw-before-repair | `query_suite_<ds>.csv`, `baseline_confirm_candidates_<ds>.csv` |
| S9 | merge confirmed baselines (after running notebook-5 patch) | `baseline_vs_generator_confirmed_<ds>.csv` |
| S10 | R2-4: grouped/duplicate-aware surrogate split | `surrogate_split_check_<ds>.csv` |
| S11 | R2-5: end-to-end latency + revised energy accounting | `end_to_end_latency_<ds>.csv`, `energy_revised_<ds>.csv` |

**Inputs (must already exist on this machine):**
- surrogate-ready HPO logs + CatBoost models + metadata (notebooks 2/3 outputs)
- all-method CF tables (notebook 4/4b outputs)
- PatchCore confirmation CSVs (notebook 5/5b outputs, `*patchcore_evaluation.csv`)
- captured score caches `cf_*.npz` (notebook 7b/7th outputs, `captured_scores/`)
- MVTec split CSVs (`mvtec_splits/<class>/df_val.csv`, `df_final.csv`) — for S2 only

Run top-to-bottom. Sections skip gracefully (with a loud warning) if an input is missing.
All outputs go to `<ROOT>/revision_outputs/{tables,figures}` per dataset.
\n> **v2 changes:** S6 signature canonicalisation fixed (v1 reported 0% for all methods due to string-format divergence) + per-column mismatch diagnostic; S7 adds dataset-relative quantile queries (`tight_band_q`, `combo_q`); S8 uses them as fallback when the absolute confirm queries are empty (the MVTec case) and never overwrites a dataset whose confirmation is already done (protects Proditec). Re-running S7 resamples DiCE, so `query_suite_*` numbers legitimately shift slightly.\n

In [1]:
# ============================================================
# S0.1) Imports, seed, dataset registry
# ============================================================
from pathlib import Path
import json, time, warnings, random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                             f1_score, r2_score, mean_squared_error,
                             mean_absolute_error)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

CWD = Path.cwd()
print("CWD:", CWD)

# Which datasets to process (each is skipped if its files are not found).
RUN_DATASETS = ["mvtec", "proditec"]

# Filenames exactly as produced by the existing pipeline notebooks
# (verified against 7b_mvtec_revision_analysis / 7th_proditec_revision_analysis).
DS_SPEC = {
    "mvtec": dict(
        data_filename="mvtec_4class_patchcore_surrogate_dataset.csv",
        model_filename="mvtec_surrogate_catboost.cbm",
        meta_filename="mvtec_surrogate_metadata.json",
        cf_subdir="mvtec_counterfactuals",
        cfs_filename="mvtec_all_methods_cfs.csv",
        rev_subdir="mvtec_revision_analysis",
        splits_subdir="mvtec_splits",
        classes=["bottle", "cable", "capsule", "hazelnut"],
    ),
    "proditec": dict(
        data_filename="patchcore_surrogate_dataset_xgb.csv",
        model_filename="proditec_surrogate_catboost.cbm",
        meta_filename="proditec_surrogate_metadata.json",
        cf_subdir="proditec_counterfactuals",
        cfs_filename="proditec_all_methods_cfs.csv",
        rev_subdir="proditec_revision_analysis",
        splits_subdir="patchcore_splits",
        classes=None,   # single product; no per-category breakdown
    ),
}

TARGET = "value"
METHOD_ORDER = ["ACFX", "CFNOW", "DiCE"]
METHOD_COLORS = {"ACFX": "tab:blue", "CFNOW": "tab:orange", "DiCE": "tab:green",
                 "LogTopK": "tab:red", "NearestFeasible": "tab:purple"}
METHOD_MARKERS = {"ACFX": "o", "CFNOW": "s", "DiCE": "^",
                  "LogTopK": "D", "NearestFeasible": "v"}

FALLBACK_FEATURES = [
    "params_backbone", "params_batch_size", "params_center_crop_key",
    "params_coreset_sampling_ratio", "params_image_size_key", "params_layers_key",
    "params_max_patches_per_image", "params_num_neighbors", "params_pre_trained",
    "params_reduction", "params_soft_corruption_level",
    "params_soft_review_budget", "params_soft_train_fraction",
]
FALLBACK_CATEGORICAL = [
    "params_backbone", "params_batch_size", "params_center_crop_key",
    "params_image_size_key", "params_layers_key", "params_max_patches_per_image",
    "params_pre_trained", "params_reduction", "params_soft_corruption_level",
]

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)


CWD: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset


In [2]:
# ============================================================
# S0.2) Robust path resolution + per-dataset context loading
#       (search strategy identical to notebooks 7b/7th)
# ============================================================

def find_files(filename):
    hits, seen, out = [], set(), []
    for base in [CWD, CWD.parent]:
        try:
            hits.extend(base.rglob(filename))
        except Exception:
            pass
    for p in hits:
        p = p.resolve()
        if p not in seen:
            seen.add(p); out.append(p)
    return out


def load_context(ds):
    spec = DS_SPEC[ds]
    hits = find_files(spec["data_filename"])
    if not hits:
        print(f"[{ds}] surrogate dataset not found -> dataset skipped."); return None
    csv_path = hits[0]
    root = csv_path.parent.parent
    art = root / "surrogate_pkl_cfs_metadata"

    ctx = dict(ds=ds, root=root, art=art)
    ctx["out_tables"] = root / "revision_outputs" / "tables"
    ctx["out_figures"] = root / "revision_outputs" / "figures"
    for d in [ctx["out_tables"], ctx["out_figures"]]:
        d.mkdir(parents=True, exist_ok=True)

    # schema
    meta_path = art / spec["meta_filename"]
    if meta_path.exists():
        m = json.load(open(meta_path))
        ctx["FEATURES"] = m["features"]
        ctx["CATEGORICAL"] = m["categorical_features"]
    else:
        ctx["FEATURES"] = list(FALLBACK_FEATURES)
        ctx["CATEGORICAL"] = list(FALLBACK_CATEGORICAL)
    ctx["NUMERIC"] = [c for c in ctx["FEATURES"] if c not in ctx["CATEGORICAL"]]

    def coerce(frame):
        x = frame.copy()
        for c in ctx["CATEGORICAL"]:
            if c in x.columns: x[c] = x[c].astype(str)
        for c in ctx["NUMERIC"]:
            if c in x.columns: x[c] = pd.to_numeric(x[c], errors="coerce")
        return x
    ctx["coerce"] = coerce

    # HPO log
    df = pd.read_csv(csv_path)
    df = coerce(df[[TARGET] + ctx["FEATURES"]])
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    ctx["df"] = df.dropna(subset=[TARGET] + ctx["FEATURES"]).reset_index(drop=True)

    # surrogate
    cb = CatBoostRegressor(); cb.load_model(str(art / spec["model_filename"]))
    ctx["cb"] = cb
    ctx["predict"] = lambda X: np.asarray(cb.predict(coerce(X[ctx["FEATURES"]])), dtype=float)

    # all-method CFs (notebook 4/4b)
    cfs_path = art / spec["cf_subdir"] / spec["cfs_filename"]
    if cfs_path.exists():
        cfs = coerce(pd.read_csv(cfs_path))
        if "cf_id" not in cfs.columns and "global_cf_id" in cfs.columns:
            cfs["cf_id"] = pd.to_numeric(cfs["global_cf_id"], errors="coerce").astype("Int64")
        ctx["cfs"] = cfs
    else:
        print(f"[{ds}] all-method CF table missing at {cfs_path}"); ctx["cfs"] = pd.DataFrame()

    # PatchCore confirmation CSV (notebook 5/5b) — glob by suffix like 7b does
    eval_dir = art / f"{spec['cf_subdir']}_evaluation"
    cands = sorted(eval_dir.glob("*patchcore_evaluation.csv")) if eval_dir.exists() else []
    if not cands:
        cands = [p for p in find_files("*patchcore_evaluation.csv")
                 if root in p.parents]
    cands = [p for p in cands if "_summary" not in p.stem and "_metadata" not in p.stem]
    if cands:
        ev = coerce(pd.read_csv(cands[0]))
        ctx["ok_df"] = (ev[ev.get("status", "ok") == "ok"].copy().reset_index(drop=True)
                        if "status" in ev.columns else ev.reset_index(drop=True))
        print(f"[{ds}] confirmation CSV: {cands[0].name}  (n_ok={len(ctx['ok_df'])})")
    else:
        ctx["ok_df"] = pd.DataFrame()
        print(f"[{ds}] WARNING: no *patchcore_evaluation.csv found")

    # captured raw-score cache (notebook 7b/7th)
    ctx["scores_dir"] = art / spec["rev_subdir"] / "captured_scores"
    npz = sorted(ctx["scores_dir"].glob("cf_*.npz")) if ctx["scores_dir"].exists() else []
    ctx["npz_files"] = npz
    print(f"[{ds}] root={root}")
    print(f"[{ds}] log={ctx['df'].shape}  cfs={ctx['cfs'].shape}  npz_cached={len(npz)}")
    return ctx


def save_table(ctx, table, stem, ff="%.4f"):
    csv_p = ctx["out_tables"] / f"{stem}.csv"
    tex_p = ctx["out_tables"] / f"{stem}.tex"
    table.to_csv(csv_p, index=False)
    try:
        table.to_latex(tex_p, index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", csv_p)
    return csv_p


def save_fig(ctx, fig, stem):
    for ext in ["pdf", "png"]:
        fig.savefig(ctx["out_figures"] / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", ctx["out_figures"] / f"{stem}.pdf")


def threshold_from_review_budget(y_score, budget):
    # identical to notebooks 5/5b/7b/7th
    assert 0.0 < budget < 1.0
    n = y_score.shape[0]
    k = max(1, int(np.ceil(budget * n)))
    return float(np.partition(y_score, -k)[-k])


CTX = {}
for _ds in RUN_DATASETS:
    c = load_context(_ds)
    if c is not None:
        CTX[_ds] = c
if not CTX:
    raise RuntimeError("No dataset contexts could be loaded — check paths above.")


[mvtec] confirmation CSV: mvtec_all_methods_cfs_patchcore_evaluation.csv  (n_ok=24)
[mvtec] root=/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class
[mvtec] log=(9976, 14)  cfs=(24, 30)  npz_cached=24
[proditec] confirmation CSV: mvtec_all_methods_cfs_patchcore_evaluation.csv  (n_ok=24)
[proditec] root=/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset
[proditec] log=(2999, 14)  cfs=(24, 30)  npz_cached=24


## S1 — Decomposition of the surrogate→confirmed gap (Reviewer 2, item 1)

The surrogate predicts **validation** F1; the paper's Table 4 gap compares it against
**test** F1 at the validation-calibrated threshold, conflating three components.
From the cached raw scores we compute, for every confirmed candidate:

- `f1_val` — actual post-retraining **validation** F1 under the candidate's own review budget
  (threshold calibrated on validation, applied to validation — the exact quantity the
  surrogate is trained to predict);
- `f1_test_deployed` — test F1 at that same validation-calibrated threshold (paper protocol);
- `f1_test_oracle` — test F1 at the best single global threshold on the test scores
  (**diagnostic upper bound on score separability, not a deployable number**);
- component split: `surrogate_error = f1_val − surrogate_pred`,
  `transfer_loss = f1_test_deployed − f1_val`, and their sum = the paper's reported gap.

Also recomputes every quantity in the paper's Table 5 so all revised numbers trace here.

In [3]:
# ============================================================
# S1) Gap decomposition from cached raw scores (no retraining)
# ============================================================

def best_global_threshold_f1(y_true, y_score):
    # Fast oracle: evaluate F1 at every distinct score cut (descending).
    order = np.argsort(-y_score)
    yt = y_true[order].astype(int)
    P = int(yt.sum())
    tp = np.cumsum(yt)
    fp = np.cumsum(1 - yt)
    fn = P - tp
    f1 = np.where((2 * tp + fp + fn) > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
    k = int(np.argmax(f1))
    return float(f1[k]), float(y_score[order][k])


DECOMP = {}
for ds, ctx in CTX.items():
    if not ctx["npz_files"]:
        print(f"[{ds}] SKIP S1 — no cached npz scores. Run notebook 7b/7th cell B3c first.")
        continue
    ok = ctx["ok_df"]
    sur_by_id = {}
    if not ok.empty and "surrogate_pred" in ok.columns and "cf_id" in ok.columns:
        sur_by_id = dict(zip(pd.to_numeric(ok["cf_id"]).astype(int),
                             pd.to_numeric(ok["surrogate_pred"], errors="coerce")))
    meth_by_id = {}
    if not ok.empty and "method" in ok.columns:
        meth_by_id = dict(zip(pd.to_numeric(ok["cf_id"]).astype(int), ok["method"].astype(str)))

    rows = []
    for p in ctx["npz_files"]:
        cf_id = int(p.stem.split("_")[1])
        z = np.load(p)
        ytv, ysv = z["y_true_val"].astype(int), z["y_score_val"].astype(float)
        ytt, yst = z["y_true_test"].astype(int), z["y_score_test"].astype(float)
        budget = float(z["budget"][0])

        thr = threshold_from_review_budget(ysv, budget)
        f1_val = f1_score(ytv, (ysv >= thr).astype(int), zero_division=0)
        f1_test = f1_score(ytt, (yst >= thr).astype(int), zero_division=0)
        f1_oracle, thr_oracle = best_global_threshold_f1(ytt, yst)
        try:  auroc_val = roc_auc_score(ytv, ysv)
        except ValueError: auroc_val = np.nan
        try:  auroc_test = roc_auc_score(ytt, yst)
        except ValueError: auroc_test = np.nan

        sp = float(sur_by_id.get(cf_id, np.nan))
        rows.append(dict(
            cf_id=cf_id, method=meth_by_id.get(cf_id, "Unknown"), budget=budget,
            surrogate_pred=sp,
            f1_val=f1_val,
            f1_test_deployed=f1_test,
            f1_test_oracle=f1_oracle,
            auroc_val=auroc_val, auroc_test=auroc_test,
            surrogate_error=f1_val - sp,
            transfer_loss=f1_test - f1_val,
            total_gap=f1_test - sp,
            threshold_transfer_loss_oracle=f1_oracle - f1_test,
        ))
    d = pd.DataFrame(rows).sort_values("cf_id").reset_index(drop=True)
    DECOMP[ds] = d

    print("=" * 66)
    print(f"[{ds}] gap decomposition over {len(d)} confirmed candidates")
    print("=" * 66)
    agg_all = d[["surrogate_pred", "f1_val", "f1_test_deployed", "f1_test_oracle",
                 "surrogate_error", "transfer_loss", "total_gap"]].agg(["mean", "std"]).T
    display(agg_all.round(4))

    per_m = (d.groupby("method")
              .agg(n=("cf_id", "size"),
                   surrogate_pred=("surrogate_pred", "mean"),
                   f1_val=("f1_val", "mean"),
                   f1_test_deployed=("f1_test_deployed", "mean"),
                   surrogate_error_mean=("surrogate_error", "mean"),
                   surrogate_error_sd=("surrogate_error", "std"),
                   transfer_loss_mean=("transfer_loss", "mean"),
                   transfer_loss_sd=("transfer_loss", "std"),
                   total_gap_mean=("total_gap", "mean"))
              .reset_index())
    display(per_m.round(4))
    save_table(ctx, d.round(6), f"decomposition_per_candidate_{ds}")
    save_table(ctx, per_m.round(4), f"decomposition_per_method_{ds}")

    # Sanity vs the paper's headline numbers
    print(f"[{ds}] mean total gap (test − surrogate): {d['total_gap'].mean():+.4f}"
          f"   (paper: −0.190 MVTec / −0.024 Proditec)")
    print(f"[{ds}] mean surrogate error (val − surrogate): {d['surrogate_error'].mean():+.4f}"
          f"   <- the part attributable to the surrogate itself")
    print(f"[{ds}] mean transfer loss (test − val):        {d['transfer_loss'].mean():+.4f}"
          f"   <- the part attributable to val→test threshold transfer")

    # Decomposition figure: stacked components per method
    fig, ax = plt.subplots(figsize=SINGLE_COL)
    mp = per_m.set_index("method").reindex([m for m in METHOD_ORDER
                                            if m in per_m["method"].values])
    x = np.arange(len(mp))
    ax.bar(x, mp["surrogate_error_mean"], 0.55, label="surrogate error (val−pred)",
           color="tab:blue", alpha=0.9)
    ax.bar(x, mp["transfer_loss_mean"], 0.55, bottom=mp["surrogate_error_mean"],
           label="threshold transfer (test−val)", color="tab:red", alpha=0.9)
    ax.scatter(x, mp["total_gap_mean"], marker="_", s=380, color="black",
               label="total gap (test−pred)", zorder=5)
    ax.axhline(0, lw=0.6, color="black")
    ax.set_xticks(x); ax.set_xticklabels(mp.index)
    ax.set_ylabel("Mean F1 gap component")
    ax.legend(frameon=False, fontsize=6)
    save_fig(ctx, fig, f"gap_decomposition_{ds}")


[mvtec] gap decomposition over 24 confirmed candidates


,mean,std
surrogate_pred,0.7890,0.0232
f1_val,0.5854,0.0232
f1_test_deployed,0.5987,0.0278
f1_test_oracle,0.8318,0.0055
surrogate_error,-0.2036,0.0408
transfer_loss,0.0133,0.0102
total_gap,-0.1903,0.0434


,method,n,surrogate_pred,f1_val,f1_test_deployed,surrogate_error_mean,surrogate_error_sd,transfer_loss_mean,transfer_loss_sd,total_gap_mean
0,ACFX,8,0.7789,0.5843,0.6035,-0.1945,0.0187,0.0191,0.0125,-0.1754
1,CFNOW,8,0.8150,0.5622,0.5690,-0.2528,0.0073,0.0069,0.0053,-0.2460
2,DiCE,8,0.7732,0.6097,0.6237,-0.1635,0.0195,0.0140,0.0086,-0.1495


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/decomposition_per_candidate_mvtec.csv
Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/decomposition_per_method_mvtec.csv
[mvtec] mean total gap (test − surrogate): -0.1903   (paper: −0.190 MVTec / −0.024 Proditec)
[mvtec] mean surrogate error (val − surrogate): -0.2036   <- the part attributable to the surrogate itself
[mvtec] mean transfer loss (test − val):        +0.0133   <- the part attributable to val→test threshold transfer
Saved figure: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/figures/gap_decomposition_mvtec.pdf
[proditec] gap decomposition over 24 confirmed candidates


,mean,std
surrogate_pred,0.8390,0.0459
f1_val,0.8299,0.0594
f1_test_deployed,0.8146,0.0643
f1_test_oracle,0.8390,0.0469
surrogate_error,-0.0091,0.0312
transfer_loss,-0.0153,0.0203
total_gap,-0.0244,0.0347


,method,n,surrogate_pred,f1_val,f1_test_deployed,surrogate_error_mean,surrogate_error_sd,transfer_loss_mean,transfer_loss_sd,total_gap_mean
0,ACFX,8,0.8015,0.7814,0.7507,-0.0201,0.0175,-0.0307,0.0119,-0.0508
1,CFNOW,8,0.8972,0.8975,0.8866,0.0003,0.0033,-0.0109,0.0044,-0.0106
2,DiCE,8,0.8182,0.8107,0.8064,-0.0075,0.0513,-0.0042,0.0276,-0.0117


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/decomposition_per_candidate_proditec.csv
Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/decomposition_per_method_proditec.csv
[proditec] mean total gap (test − surrogate): -0.0244   (paper: −0.190 MVTec / −0.024 Proditec)
[proditec] mean surrogate error (val − surrogate): -0.0091   <- the part attributable to the surrogate itself
[proditec] mean transfer loss (test − val):        -0.0153   <- the part attributable to val→test threshold transfer
Saved figure: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/figures/gap_decomposition_proditec.pdf


## S2 — Two distinct mechanisms: pooling→ranking (AUROC) vs threshold→F1 (Reviewer 2, item 1)

AUROC is threshold-independent, so the revised text must attribute:
1. **pooled-AUROC collapse** to *pooling incompatible per-category score scales*
   (a ranking effect, no threshold involved), quantified as
   `pooling_penalty = mean per-category AUROC − pooled AUROC`;
2. **F1 loss** to the *single pooled validation-calibrated threshold*
   (a thresholding effect), quantified as
   `f1_gain_classwise = F1@class-wise thresholds − F1@pooled threshold`.

MVTec only (Proditec is a single product). Requires split CSVs for class labels.

In [4]:
# ============================================================
# S2) Pooling (ranking/AUROC) vs thresholding (F1) — MVTec
# ============================================================
ds = "mvtec"
if ds not in CTX or not CTX[ds]["npz_files"]:
    print("SKIP S2 — MVTec context or npz cache unavailable.")
else:
    ctx = CTX[ds]; spec = DS_SPEC[ds]
    splits_dir = ctx["root"] / spec["splits_subdir"]

    def load_class_labels(split_csv_name):
        parts = []
        for cls in spec["classes"]:
            pth = splits_dir / cls / split_csv_name
            if not pth.exists():
                raise FileNotFoundError(pth)
            d = pd.read_csv(pth)
            if "class_name" not in d.columns:
                d["class_name"] = cls
            parts.append(d[["class_name"]])
        return pd.concat(parts, ignore_index=True)["class_name"].astype(str).to_numpy()

    try:
        VAL_CLASS = load_class_labels("df_val.csv")
        TEST_CLASS = load_class_labels("df_final.csv")
    except FileNotFoundError as e:
        VAL_CLASS = TEST_CLASS = None
        print("SKIP S2 — split CSV missing:", e)

    if VAL_CLASS is not None:
        rows = []
        for p in ctx["npz_files"]:
            cf_id = int(p.stem.split("_")[1])
            z = np.load(p)
            ytv, ysv = z["y_true_val"].astype(int), z["y_score_val"].astype(float)
            ytt, yst = z["y_true_test"].astype(int), z["y_score_test"].astype(float)
            budget = float(z["budget"][0])
            if len(ytv) != len(VAL_CLASS) or len(ytt) != len(TEST_CLASS):
                print(f"CF {cf_id}: split-length mismatch; skipped."); continue

            thr_pooled = threshold_from_review_budget(ysv, budget)
            pred_pooled = (yst >= thr_pooled).astype(int)
            try:  auroc_pooled = roc_auc_score(ytt, yst)
            except ValueError: auroc_pooled = np.nan

            per_auc, f1_pooled_cls, f1_cw_cls = [], [], []
            for cls in np.unique(TEST_CLASS):
                mt, mv = (TEST_CLASS == cls), (VAL_CLASS == cls)
                try:  per_auc.append(roc_auc_score(ytt[mt], yst[mt]))
                except ValueError: per_auc.append(np.nan)
                f1_pooled_cls.append(f1_score(ytt[mt], pred_pooled[mt], zero_division=0))
                thr_c = threshold_from_review_budget(ysv[mv], budget)
                f1_cw_cls.append(f1_score(ytt[mt], (yst[mt] >= thr_c).astype(int),
                                          zero_division=0))
            rows.append(dict(
                cf_id=cf_id,
                auroc_pooled=auroc_pooled,
                auroc_per_class_mean=float(np.nanmean(per_auc)),
                f1_pooled_thr=float(np.mean(f1_pooled_cls)),
                f1_classwise_thr=float(np.mean(f1_cw_cls)),
            ))
        t = pd.DataFrame(rows)
        t["pooling_penalty_ranking"] = t["auroc_per_class_mean"] - t["auroc_pooled"]
        t["f1_gain_classwise_threshold"] = t["f1_classwise_thr"] - t["f1_pooled_thr"]
        display(t.round(4).head(8))
        summ = pd.DataFrame([dict(
            mechanism="1. Pooling of category score scales (ranking effect, threshold-free)",
            metric="mean per-category AUROC − pooled AUROC",
            value=float(t["pooling_penalty_ranking"].mean()),
        ), dict(
            mechanism="2. Single pooled validation-calibrated threshold (F1 effect)",
            metric="mean F1 @ class-wise thresholds − F1 @ pooled threshold",
            value=float(t["f1_gain_classwise_threshold"].mean()),
        )])
        display(summ.round(4))
        save_table(ctx, t.round(4), "two_mechanisms_per_candidate_mvtec")
        save_table(ctx, summ.round(4), "two_mechanisms_summary_mvtec")
        print("\nRevised-text wording anchor:")
        print("  Pooling incompatible score scales damages the *global ranking* "
              f"(AUROC penalty {t['pooling_penalty_ranking'].mean():.3f});")
        print("  the single pooled threshold *additionally* damages F1 "
              f"(class-wise recovery {t['f1_gain_classwise_threshold'].mean():.3f}).")
        print("  Paper reference values: penalty ~0.330, F1 gain ~0.248.")


,cf_id,auroc_pooled,auroc_per_class_mean,f1_pooled_thr,f1_classwise_thr,pooling_penalty_ranking,f1_gain_classwise_threshold
0,0,0.5022,0.8847,0.5186,0.7775,0.3825,0.2589
1,1,0.4873,0.7677,0.5960,0.7666,0.2804,0.1706
2,10,0.5341,0.9086,0.4275,0.7583,0.3745,0.3308
3,11,0.5345,0.9152,0.4260,0.7706,0.3807,0.3446
4,12,0.5083,0.8990,0.4260,0.7982,0.3907,0.3722
5,13,0.5242,0.9032,0.4260,0.7750,0.3791,0.3489
6,14,0.5210,0.8912,0.4289,0.7608,0.3703,0.3318
7,15,0.5220,0.8985,0.4260,0.7750,0.3765,0.3489


,mechanism,metric,value
0,1. Pooling of category score scales (ranking e...,mean per-category AUROC − pooled AUROC,0.3298
1,2. Single pooled validation-calibrated thresho...,mean F1 @ class-wise thresholds − F1 @ pooled ...,0.2482


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/two_mechanisms_per_candidate_mvtec.csv
Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/two_mechanisms_summary_mvtec.csv

Revised-text wording anchor:
  Pooling incompatible score scales damages the *global ranking* (AUROC penalty 0.330);
  the single pooled threshold *additionally* damages F1 (class-wise recovery 0.248).
  Paper reference values: penalty ~0.330, F1 gain ~0.248.


## S3 — Table 4 with dispersion (both reviews)

Means over n=8 per method hide dispersion. Regenerate Table 4 with SD and min–max,
and flag the §4.7 within-generator correlations (n=8) as descriptive-only.

In [5]:
# ============================================================
# S3) Table-4 variability + descriptive-correlation caveat
# ============================================================
from scipy import stats

for ds, ctx in CTX.items():
    ok = ctx["ok_df"]
    if ok.empty:
        print(f"[{ds}] SKIP S3 — no confirmation table."); continue
    need = ["method", "surrogate_pred", "f1_test"]
    if any(c not in ok.columns for c in need):
        print(f"[{ds}] SKIP S3 — missing columns {need}"); continue
    g = ok.copy()
    g["method"] = g["method"].astype(str)
    gap_col = ("gap_test_minus_surrogate_pred"
               if "gap_test_minus_surrogate_pred" in g.columns else None)
    if gap_col is None:
        g["gap_test_minus_surrogate_pred"] = (
            pd.to_numeric(g["f1_test"]) - pd.to_numeric(g["surrogate_pred"]))
        gap_col = "gap_test_minus_surrogate_pred"

    def agg(col):
        s = pd.to_numeric(g.groupby("method")[col].mean(), errors="coerce")
        return s
    tab = pd.DataFrame({"method": sorted(g["method"].unique())}).set_index("method")
    for col, label in [("distance_to_factual", "distance"),
                       ("generation_sec", "gen_time_s"),
                       ("surrogate_pred", "surrogate_f1"),
                       ("f1_test", "patchcore_f1"),
                       (gap_col, "gap")]:
        if col not in g.columns:
            continue
        v = g.groupby("method")[col]
        tab[f"{label}_mean"] = v.mean()
        tab[f"{label}_sd"] = v.std()
        tab[f"{label}_min"] = v.min()
        tab[f"{label}_max"] = v.max()
    tab = tab.reset_index()
    display(tab.round(4))
    save_table(ctx, tab.round(4), f"table4_variability_{ds}")

    # Descriptive within-method correlations (explicit n and caveat)
    if "distance_to_factual" in g.columns:
        rows = []
        for name, grp in [("all", g)] + list(g.groupby("method")):
            if len(grp) < 3: continue
            r, p = stats.pearsonr(grp["distance_to_factual"], grp[gap_col])
            rows.append(dict(scope=name, n=len(grp), pearson_r=r, pearson_p=p,
                             note="descriptive only (n<=8 per method)"
                                  if name != "all" else ""))
        ct = pd.DataFrame(rows)
        display(ct.round(4))
        save_table(ctx, ct.round(4), f"gap_distance_correlations_descriptive_{ds}")


,method,distance_mean,distance_sd,distance_min,distance_max,gen_time_s_mean,gen_time_s_sd,gen_time_s_min,gen_time_s_max,surrogate_f1_mean,surrogate_f1_sd,surrogate_f1_min,surrogate_f1_max,patchcore_f1_mean,patchcore_f1_sd,patchcore_f1_min,patchcore_f1_max,gap_mean,gap_sd,gap_min,gap_max
0,ACFX,0.9083,0.2715,0.6030,1.4250,99.8369,0.0,99.8369,99.8369,0.7789,0.0208,0.7396,0.7948,0.6067,0.0229,0.5645,0.6250,-0.1722,0.0112,-0.1890,-0.1559
1,CFNOW,5.6502,0.4948,4.8529,6.0780,5.1358,0.0,5.1358,5.1358,0.8150,0.0057,0.8083,0.8228,0.5687,0.0029,0.5664,0.5754,-0.2463,0.0050,-0.2534,-0.2399
2,DiCE,1.5831,0.6252,0.4525,2.4364,0.3643,0.0,0.3643,0.3643,0.7732,0.0114,0.7603,0.7948,0.6229,0.0167,0.5965,0.6389,-0.1503,0.0187,-0.1867,-0.1296


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/table4_variability_mvtec.csv


,scope,n,pearson_r,pearson_p,note
0,all,24,-0.9051,0.0000,
1,ACFX,8,-0.4598,0.2517,descriptive only (n<=8 per method)
2,CFNOW,8,-0.0162,0.9697,descriptive only (n<=8 per method)
3,DiCE,8,-0.5392,0.1678,descriptive only (n<=8 per method)


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/gap_distance_correlations_descriptive_mvtec.csv


,method,distance_mean,distance_sd,distance_min,distance_max,gen_time_s_mean,gen_time_s_sd,gen_time_s_min,gen_time_s_max,surrogate_f1_mean,surrogate_f1_sd,surrogate_f1_min,surrogate_f1_max,patchcore_f1_mean,patchcore_f1_sd,patchcore_f1_min,patchcore_f1_max,gap_mean,gap_sd,gap_min,gap_max
0,ACFX,0.7957,0.2302,0.4424,1.1087,111.3662,0.0,111.3662,111.3662,0.8015,0.0290,0.7556,0.8342,0.7507,0.0262,0.7056,0.7931,-0.0508,0.0240,-0.0833,-0.0049
1,CFNOW,3.8190,0.0322,3.7700,3.8590,1.4920,0.0,1.4920,1.4920,0.8972,0.0003,0.8968,0.8978,0.8866,0.0052,0.8823,0.8955,-0.0106,0.0052,-0.0148,-0.0016
2,DiCE,1.5522,1.0697,0.4532,3.6795,0.3367,0.0,0.3367,0.3367,0.8182,0.0112,0.8061,0.8384,0.8064,0.0470,0.7423,0.8681,-0.0117,0.0463,-0.0741,0.0472


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/table4_variability_proditec.csv


,scope,n,pearson_r,pearson_p,note
0,all,24,0.5733,0.0034,
1,ACFX,8,0.2570,0.5389,descriptive only (n<=8 per method)
2,CFNOW,8,0.7415,0.0352,descriptive only (n<=8 per method)
3,DiCE,8,0.6917,0.0574,descriptive only (n<=8 per method)


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/gap_distance_correlations_descriptive_proditec.csv


## S4 — Surrogate-vs-confirmed scatter for **both** datasets (R2 minor)

The paper shows this only for Proditec (Fig. 2); add the MVTec twin.

In [6]:
# ============================================================
# S4) Confirmed PatchCore F1 vs surrogate F1 — both datasets
# ============================================================
for ds, ctx in CTX.items():
    ok = ctx["ok_df"]
    if ok.empty or "surrogate_pred" not in ok.columns or "f1_test" not in ok.columns:
        print(f"[{ds}] SKIP S4"); continue
    g = ok.copy(); g["method"] = g.get("method", "All").astype(str)
    fig, ax = plt.subplots(figsize=SINGLE_COL)
    lo = float(min(g["surrogate_pred"].min(), g["f1_test"].min())) - 0.02
    hi = float(max(g["surrogate_pred"].max(), g["f1_test"].max())) + 0.02
    ax.plot([lo, hi], [lo, hi], "--", lw=0.8, color="tab:blue",
            label="Ideal agreement")
    for m in sorted(g["method"].unique()):
        sub = g[g["method"] == m]
        ax.scatter(sub["surrogate_pred"], sub["f1_test"], s=26, alpha=0.85,
                   marker=METHOD_MARKERS.get(m, "o"),
                   color=METHOD_COLORS.get(m), label=f"{m} (n={len(sub)})")
    ax.set_xlabel("Surrogate F1"); ax.set_ylabel("Confirmed PatchCore F1")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.legend(frameon=False, fontsize=6)
    save_fig(ctx, fig, f"surrogate_vs_confirmed_scatter_{ds}")


Saved figure: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/figures/surrogate_vs_confirmed_scatter_mvtec.pdf
Saved figure: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/figures/surrogate_vs_confirmed_scatter_proditec.pdf


## S5 — Multivariate check of the "corruption is the dominant driver" claim (R2, item 8)

The paper's top-6-vs-18 split is post hoc. Here: standardized OLS of confirmed F1 on the
three soft controls jointly, plus per-control Spearman, over the 24 confirmed candidates.
Framed as descriptive support for a **softened** claim.

In [7]:
# ============================================================
# S5) Soft-control regression on confirmed F1 (descriptive)
# ============================================================
try:
    import statsmodels.api as sm
    HAS_SM = True
except Exception:
    HAS_SM = False
    print("statsmodels unavailable -> coefficients via sklearn, no p-values")
from sklearn.linear_model import LinearRegression

for ds, ctx in CTX.items():
    ok = ctx["ok_df"]
    need = ["f1_test", "params_soft_train_fraction",
            "params_soft_review_budget", "params_soft_corruption_level"]
    if ok.empty or any(c not in ok.columns for c in need):
        print(f"[{ds}] SKIP S5 — confirmation CSV lacks soft-control columns."); continue
    g = ok.copy()
    y = pd.to_numeric(g["f1_test"], errors="coerce")
    X = pd.DataFrame({
        "train_fraction": pd.to_numeric(g["params_soft_train_fraction"], errors="coerce"),
        "review_budget": pd.to_numeric(g["params_soft_review_budget"], errors="coerce"),
        "corruption_mild": (g["params_soft_corruption_level"].astype(str) == "mild").astype(float),
        "corruption_strong": (g["params_soft_corruption_level"].astype(str) == "strong").astype(float),
    })
    Z = (X - X.mean()) / X.std(ddof=0).replace(0, 1.0)
    mask = y.notna() & Z.notna().all(axis=1)
    if HAS_SM:
        Zc = sm.add_constant(Z[mask])
        res = sm.OLS(y[mask], Zc).fit()
        coef = pd.DataFrame({"coef_std": res.params, "p": res.pvalues}).drop(
            index="const", errors="ignore")
        coef["R2"] = res.rsquared
    else:
        lr = LinearRegression().fit(Z[mask], y[mask])
        coef = pd.DataFrame({"coef_std": lr.coef_, "p": np.nan},
                            index=Z.columns)
        coef["R2"] = lr.score(Z[mask], y[mask])
    coef["n"] = int(mask.sum())
    display(coef.round(4))
    sp = {c: stats.spearmanr(X[c], y, nan_policy="omit") for c in X.columns}
    sp_t = pd.DataFrame([dict(control=k, spearman_rho=v.correlation, p=v.pvalue)
                         for k, v in sp.items()])
    display(sp_t.round(4))
    out = coef.reset_index().rename(columns={"index": "control"})
    save_table(ctx, out.round(4), f"soft_control_regression_{ds}")
    save_table(ctx, sp_t.round(4), f"soft_control_spearman_{ds}")
    print(f"[{ds}] n={len(g)}: treat as descriptive; use to justify the softened wording "
          "'the most visible operator-facing pattern' rather than 'dominant driver'.")


,coef_std,p,R2,n
train_fraction,-0.0030,0.5877,0.4887,24
review_budget,0.0024,0.6559,0.4887,24
corruption_mild,0.0186,0.0011,0.4887,24
corruption_strong,0.0000,NaN,0.4887,24


,control,spearman_rho,p
0,train_fraction,-0.4604,0.0236
1,review_budget,-0.4078,0.0479
2,corruption_mild,0.6180,0.0013
3,corruption_strong,NaN,NaN


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/soft_control_regression_mvtec.csv
Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/soft_control_spearman_mvtec.csv
[mvtec] n=24: treat as descriptive; use to justify the softened wording 'the most visible operator-facing pattern' rather than 'dominant driver'.


,coef_std,p,R2,n
train_fraction,-0.0099,0.3079,0.6666,24
review_budget,0.0126,0.1701,0.6666,24
corruption_mild,-0.0011,0.9048,0.6666,24
corruption_strong,-0.0483,0.0001,0.6666,24


,control,spearman_rho,p
0,train_fraction,-0.3611,0.0830
1,review_budget,0.0833,0.6989
2,corruption_mild,0.1054,0.6239
3,corruption_strong,-0.7692,0.0000


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/soft_control_regression_proditec.csv
Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/soft_control_spearman_proditec.csv
[proditec] n=24: treat as descriptive; use to justify the softened wording 'the most visible operator-facing pattern' rather than 'dominant driver'.


## S6 — Duplication rate of generated CFs vs logged trials (R2 minor)

Exact-match on the full (grid-snapped) feature tuple. NOTE: **CFNOW candidates are
logged trials by construction** (notebook 4/4b implements CFNOW-style search *over the
observed HPO log*), so its rate is expected to be 100% — this must be stated openly
in the revised text, and the method relabelled as a log-retrieval baseline.

In [8]:
# ============================================================
# S6) Duplication rate vs logged configurations  (v2 — fixed)
# v1 returned 0% for ALL methods incl. CFNOW because raw
# `astype(str)` signatures diverged on formatting ("16.0" vs "16",
# float repr noise). v2 canonicalises every column to a fixed
# string form and self-diagnoses any residual mismatch.
# ============================================================

_NUM_FMT = {"params_soft_review_budget":  ("%.3f", 0.005, 0.0),
            "params_soft_train_fraction": ("%.2f", 0.01,  0.0),
            "params_coreset_sampling_ratio": ("%.3f", 0.002, 0.001),
            "params_num_neighbors": ("%d", 1.0, 0.0)}

def _canon_cat(s):
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)          # "16.0" -> "16"
    s = s.str.replace(r"^(true|false)$",
                      lambda m: m.group(1).capitalize(), regex=True)
    return s

def snap_features_frame(ctx, frame):
    x = ctx["coerce"](frame[ctx["FEATURES"]]).copy()
    for c, (fmt, step, origin) in _NUM_FMT.items():
        if c in x.columns:
            v = pd.to_numeric(x[c], errors="coerce")
            snapped = origin + np.round((v - origin) / step) * step
            x[c] = snapped
    return x

def signature_series(ctx, frame):
    x = snap_features_frame(ctx, frame)
    cols = []
    for c in ctx["FEATURES"]:
        if c in _NUM_FMT:
            fmt = _NUM_FMT[c][0]
            cols.append(pd.to_numeric(x[c], errors="coerce")
                        .map(lambda v, f=fmt: f % v if pd.notna(v) else "NA"))
        elif c in ctx["CATEGORICAL"]:
            cols.append(_canon_cat(x[c]))
        else:
            cols.append(pd.to_numeric(x[c], errors="coerce")
                        .map(lambda v: "%.6g" % v if pd.notna(v) else "NA"))
    return pd.concat(cols, axis=1).agg("|".join, axis=1)


for ds, ctx in CTX.items():
    cfs = ctx["cfs"]
    if cfs.empty:
        print(f"[{ds}] SKIP S6 — no CF table."); continue

    log_sig = signature_series(ctx, ctx["df"])
    log_set = set(log_sig)
    c = cfs.copy()
    c["in_log"] = signature_series(ctx, c).isin(log_set)
    mcol = "method" if "method" in c.columns else None
    if mcol:
        t = (c.groupby(mcol)["in_log"].agg(n="size", duplicates="sum")
              .reset_index())
        t["duplicate_rate_pct"] = 100 * t["duplicates"] / t["n"]
    else:
        t = pd.DataFrame([dict(method="all", n=len(c),
                               duplicates=int(c["in_log"].sum()),
                               duplicate_rate_pct=100 * c["in_log"].mean())])
    display(t.round(2))
    save_table(ctx, t.round(2), f"duplication_rate_{ds}")

    # ---- self-diagnostic: if CFNOW is not ~100%, show WHY column-by-column ----
    if mcol and "CFNOW" in set(c[mcol]):
        rate = float(t.loc[t[mcol] == "CFNOW", "duplicate_rate_pct"].iloc[0])
        print(f"[{ds}] CFNOW duplicate rate = {rate:.0f}% "
              "(expected ~100%: it searches the observed log by construction).")
        if rate < 99.0:
            row = snap_features_frame(ctx, c[c[mcol] == "CFNOW"].head(1))
            row_sig_cols = {f: signature_series(ctx, c[c[mcol]=="CFNOW"].head(1)
                                                ).iloc[0].split("|")[i]
                            for i, f in enumerate(ctx["FEATURES"])}
            L = snap_features_frame(ctx, ctx["df"])
            log_cols = signature_series(ctx, ctx["df"]).str.split("|", expand=True)
            log_cols.columns = ctx["FEATURES"]
            matches = (log_cols == pd.Series(row_sig_cols)).sum(axis=1)
            j = int(matches.idxmax())
            print(f"  DIAGNOSTIC vs closest log row (idx {j}, "
                  f"{int(matches.max())}/{len(ctx['FEATURES'])} fields equal):")
            for f in ctx["FEATURES"]:
                a, b = row_sig_cols[f], log_cols.loc[j, f]
                if a != b:
                    print(f"    MISMATCH {f}: cf='{a}'  vs  log='{b}'")


,method,n,duplicates,duplicate_rate_pct
0,ACFX,8,0,0.0
1,CFNOW,8,8,100.0
2,DiCE,8,0,0.0


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/duplication_rate_mvtec.csv
[mvtec] CFNOW duplicate rate = 100% (expected ~100%: it searches the observed log by construction).


,method,n,duplicates,duplicate_rate_pct
0,ACFX,8,0,0.0
1,CFNOW,8,8,100.0
2,DiCE,8,0,0.0


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/duplication_rate_proditec.csv
[proditec] CFNOW duplicate rate = 100% (expected ~100%: it searches the observed log by construction).


## S7 — Constrained query suite with raw-before-repair reporting (R2, items 2–3)

One broad free-controls query made 100% validity uninformative. This suite runs
**10 query types × 3 anchors (p50 / p85 / best)** per dataset over:

- **DiCE** (true generator, unseen configurations possible),
- **LogTopK** (constrained log filtering + surrogate-score top-K — the honest relabel of
  the paper's "CFNOW"),
- **NearestFeasible** (nearest strictly-valid logged configuration by the paper's
  mixed-type distance — R2's requested proximity-retrieval baseline).

Per (query, anchor, method) it reports: requested K, **raw candidates before any
repair/snap**, raw strictly-valid, repairs applied, final strictly-valid, shortfall
(abstention), mean surrogate F1, mean distance, wall time, and the number of feasible
logged trials (the coverage statistic that decides when generation adds value over
retrieval). ACFX can be enabled via the toggle but is off by default for cost.

Surrogate-level only — zero retraining.

In [9]:
# ============================================================
# S7.1) Query suite definition + shared machinery
# ============================================================
INCLUDE_ACFX_IN_SUITE = False   # two orders of magnitude slower; enable if desired
K_SUITE = 8
ANCHOR_QUANTILES = [0.50, 0.85]   # + best trial appended below


def mixed_distance(ctx, cand_df, anchor_row):
    # Eq. 14: sum |Δnum|/range + count categorical mismatches
    d = np.zeros(len(cand_df), dtype=float)
    for c in ctx["NUMERIC"]:
        r = float(ctx["df"][c].max() - ctx["df"][c].min()) or 1.0
        d += np.abs(pd.to_numeric(cand_df[c], errors="coerce").to_numpy()
                    - float(anchor_row[c])) / r
    for c in ctx["CATEGORICAL"]:
        d += (cand_df[c].astype(str).to_numpy() != str(anchor_row[c])).astype(float)
    return d


def resolve_query(q, ctx, anchor_row, anchor_pred):
    # objective -> [L, U]
    ob = q["objective"]
    if ob[0] == "target_band":
        L, U = ob[1], ob[2]
    elif ob[0] == "target_band_q":   # v2: dataset-relative band from log quantiles
        L = float(ctx["df"][TARGET].quantile(ob[1]))
        U = float(ctx["df"][TARGET].quantile(ob[2]))
    elif ob[0] == "target_min":
        L, U = ob[1], 1.0
    elif ob[0] == "delta_improve":
        L, U = anchor_pred + ob[1], 1.0
    # soft constraints -> concrete bounds/sets
    tf = q.get("train_fraction", ("free",))
    rb = q.get("review_budget", ("free",))
    cr = q.get("corruption", ("free",))
    def rng(spec, lo_all, hi_all, anchor_val=None):
        if spec[0] == "free":  return (lo_all, hi_all)
        if spec[0] == "range": return (max(lo_all, spec[1]), min(hi_all, spec[2]))
        if spec[0] == "max":   return (lo_all, min(hi_all, spec[1]))
        if spec[0] == "rel_max":  # e.g. budget <= factor * anchor budget
            return (lo_all, min(hi_all, spec[1] * float(anchor_val)))
        if spec[0] == "fixed": return (spec[1], spec[1])
        raise ValueError(spec)
    tf_b = rng(tf, 0.20, 1.00, anchor_row["params_soft_train_fraction"])
    rb_b = rng(rb, 0.005, 0.500, anchor_row["params_soft_review_budget"])
    if cr[0] == "free":
        cr_s = {"none", "mild", "strong"}
    elif cr[0] == "allowed":
        cr_s = set(cr[1])
    else:
        cr_s = {cr[1]}
    return dict(L=L, U=U, tf=tf_b, rb=rb_b, corr=cr_s)


def feasible_mask(ctx, frame, rq):
    tf = pd.to_numeric(frame["params_soft_train_fraction"], errors="coerce")
    rb = pd.to_numeric(frame["params_soft_review_budget"], errors="coerce")
    cl = frame["params_soft_corruption_level"].astype(str)
    return ((tf >= rq["tf"][0] - 1e-9) & (tf <= rq["tf"][1] + 1e-9)
            & (rb >= rq["rb"][0] - 1e-9) & (rb <= rq["rb"][1] + 1e-9)
            & cl.isin(rq["corr"]))


QUERY_SUITE = [
    dict(qid="broad_band_free", objective=("target_band", 0.75, 0.95)),
    dict(qid="tight_band", objective=("target_band", 0.85, 0.90)),
    dict(qid="target_min_085", objective=("target_min", 0.85)),
    dict(qid="delta_improve_005", objective=("delta_improve", 0.05)),
    dict(qid="low_budget", objective=("target_band", 0.75, 0.95),
         review_budget=("max", 0.30)),
    dict(qid="low_train_fraction", objective=("target_band", 0.75, 0.95),
         train_fraction=("max", 0.60)),
    dict(qid="corruption_none_mild", objective=("target_band", 0.75, 0.95),
         corruption=("allowed", ["none", "mild"])),
    dict(qid="relative_budget_090", objective=("delta_improve", 0.0),
         review_budget=("rel_max", 0.90)),
    dict(qid="combo_tight", objective=("target_band", 0.85, 0.92),
         review_budget=("max", 0.30), train_fraction=("max", 0.60),
         corruption=("allowed", ["none", "mild"])),
    dict(qid="infeasible_high", objective=("target_min", 0.99)),
]


# ---- v2: dataset-relative (quantile-based) queries -------------------------
# Absolute bands like [0.85, 0.90] can be empty/infeasible on datasets whose
# target distribution sits lower (MVTec pooled-F1). These resolve per dataset
# at runtime inside resolve_query via ("target_band_q", qlo, qhi).
QUERY_SUITE += [
    dict(qid="tight_band_q", objective=("target_band_q", 0.80, 0.95)),
    dict(qid="combo_q", objective=("target_band_q", 0.80, 0.95),
         review_budget=("max", 0.30), train_fraction=("max", 0.60),
         corruption=("allowed", ["none", "mild"])),
]
print(f"{len(QUERY_SUITE)} queries defined (incl. quantile-based).")


12 queries defined (incl. quantile-based).


In [10]:
# ============================================================
# S7.2) Generator + retrieval implementations for the suite
# ============================================================
import dice_ml
from dice_ml import Dice

SUITE_CTX = {}
for ds, ctx in CTX.items():
    df = ctx["df"]
    # anchors: p50, p85, best (subset of the paper's 5-anchor sweep, bounded runtime)
    idxs, labels = [], []
    for q in ANCHOR_QUANTILES:
        tv = df[TARGET].quantile(q)
        idxs.append(int((df[TARGET] - tv).abs().idxmin())); labels.append(f"p{int(q*100)}")
    idxs.append(int(df[TARGET].idxmax())); labels.append("best")
    anchors = []
    for lab, i in zip(labels, idxs):
        row = df.loc[i, ctx["FEATURES"]]
        x_df = ctx["coerce"](pd.DataFrame([row.values], columns=ctx["FEATURES"]))
        anchors.append(dict(label=lab, row=row, x_df=x_df,
                            pred=float(ctx["predict"](x_df)[0])))

    # DiCE (identical construction to notebooks 4/7)
    dd = df[ctx["FEATURES"] + [TARGET]].copy()
    for c in ctx["CATEGORICAL"]:
        dd[c] = dd[c].astype(str)
    cont = [c for c in ctx["FEATURES"] if c not in ctx["CATEGORICAL"]]
    data_dice = dice_ml.Data(dataframe=dd, continuous_features=cont, outcome_name=TARGET)
    model_dice = dice_ml.Model(model=ctx["cb"], backend="sklearn", model_type="regressor")
    exp_dice = Dice(data_dice, model_dice, method="random")

    # full-log surrogate predictions cached once (fair shared information)
    log_pred = ctx["predict"](df)

    SUITE_CTX[ds] = dict(anchors=anchors, exp_dice=exp_dice, log_pred=log_pred)
    print(f"[{ds}] suite ready — anchors: {[a['label'] for a in anchors]}")


def dice_permitted_range(ctx, rq):
    pr = {}
    for c in ctx["NUMERIC"]:
        lo, hi = float(ctx["df"][c].min()), float(ctx["df"][c].max())
        if c == "params_soft_train_fraction": lo, hi = rq["tf"]
        if c == "params_soft_review_budget":  lo, hi = rq["rb"]
        pr[c] = [lo, hi]
    for c in ctx["CATEGORICAL"]:
        vals = sorted(ctx["df"][c].astype(str).unique().tolist())
        if c == "params_soft_corruption_level":
            vals = sorted(rq["corr"])
        pr[c] = vals
    return pr


def run_dice(ctx, sc, anchor, rq, k):
    t0 = time.time()
    try:
        res = sc["exp_dice"].generate_counterfactuals(
            anchor["x_df"][ctx["FEATURES"]],
            total_CFs=max(30, 5 * k),
            desired_range=[rq["L"], rq["U"]],
            features_to_vary=ctx["FEATURES"],
            permitted_range=dice_permitted_range(ctx, rq),
        )
        ex = res.cf_examples_list[0]
        raw = (ex.final_cfs_df_sparse if getattr(ex, "final_cfs_df_sparse", None)
               is not None else ex.final_cfs_df).copy()
    except Exception as e:
        return None, time.time() - t0, repr(e)
    return raw, time.time() - t0, None


def run_log_topk(ctx, sc, anchor, rq, k):
    t0 = time.time()
    m = feasible_mask(ctx, ctx["df"], rq).to_numpy()
    pred = sc["log_pred"]
    band = (pred >= rq["L"]) & (pred <= rq["U"]) & m
    idx = np.where(band)[0]
    idx = idx[np.argsort(-pred[idx])][:k]
    out = ctx["df"].iloc[idx][ctx["FEATURES"]].copy()
    return out, time.time() - t0, None, int(m.sum()), int(band.sum())


def run_nearest_feasible(ctx, sc, anchor, rq, k):
    t0 = time.time()
    m = feasible_mask(ctx, ctx["df"], rq).to_numpy()
    pred = sc["log_pred"]
    band = (pred >= rq["L"]) & (pred <= rq["U"]) & m
    idx = np.where(band)[0]
    if len(idx):
        sub = ctx["df"].iloc[idx]
        d = mixed_distance(ctx, sub, anchor["row"])
        order = np.argsort(d)[:k]
        out = sub.iloc[order][ctx["FEATURES"]].copy()
    else:
        out = ctx["df"].iloc[[]][ctx["FEATURES"]].copy()
    return out, time.time() - t0, None


[mvtec] suite ready — anchors: ['p50', 'p85', 'best']
[proditec] suite ready — anchors: ['p50', 'p85', 'best']


In [11]:
# ============================================================
# S7.3) Execute the suite; report raw-before-repair honestly
# ============================================================

def evaluate_candidates(ctx, raw, anchor, rq, k, snap=True):
    """Return (metrics_dict, final_df). Raw metrics computed BEFORE snapping."""
    if raw is None or len(raw) == 0:
        return dict(raw_n=0, raw_valid_n=0, repairs=0, final_valid_n=0,
                    shortfall=k, mean_surrogate_f1=np.nan,
                    mean_distance=np.nan), None
    x_raw = ctx["coerce"](raw.reindex(columns=ctx["FEATURES"]))
    x_raw = x_raw.dropna(subset=ctx["FEATURES"]).reset_index(drop=True)
    raw_n = len(x_raw)
    pred_raw = ctx["predict"](x_raw) if raw_n else np.array([])
    fm_raw = feasible_mask(ctx, x_raw, rq).to_numpy() if raw_n else np.array([], bool)
    raw_valid = ((pred_raw >= rq["L"]) & (pred_raw <= rq["U"]) & fm_raw)

    if snap:
        x = snap_features_frame(ctx, x_raw)
        repairs = int((~np.isclose(
            x[ctx["NUMERIC"]].to_numpy(dtype=float),
            x_raw[ctx["NUMERIC"]].to_numpy(dtype=float),
            rtol=0, atol=1e-9, equal_nan=True)).any(axis=1).sum())
    else:
        x, repairs = x_raw, 0

    pred = ctx["predict"](x)
    fm = feasible_mask(ctx, x, rq).to_numpy()
    final_valid = (pred >= rq["L"]) & (pred <= rq["U"]) & fm
    xv = x[final_valid].copy()
    pv = pred[final_valid]
    # dedupe + top-k by predicted quality
    sig = xv.astype(str).agg("|".join, axis=1)
    keep = ~sig.duplicated()
    xv, pv = xv[keep.values], pv[keep.values]
    order = np.argsort(-pv)[:k]
    xv, pv = xv.iloc[order], pv[order]
    dist = mixed_distance(ctx, xv, anchor["row"]) if len(xv) else np.array([])

    return dict(raw_n=raw_n,
                raw_valid_n=int(raw_valid.sum()),
                repairs=repairs,
                final_valid_n=int(len(xv)),
                shortfall=max(0, k - len(xv)),
                mean_surrogate_f1=float(np.mean(pv)) if len(pv) else np.nan,
                mean_distance=float(np.mean(dist)) if len(dist) else np.nan), xv


SUITE_RESULTS = {}
SUITE_TOP = {}
for ds, ctx in CTX.items():
    sc = SUITE_CTX[ds]
    rows, top_rows = [], []
    for q in QUERY_SUITE:
        for a in sc["anchors"]:
            rq = resolve_query(q, ctx, a["row"], a["pred"])
            feas_n = int(feasible_mask(ctx, ctx["df"], rq).sum())

            jobs = [("DiCE", lambda: run_dice(ctx, sc, a, rq, K_SUITE) + (None, None)),
                    ("LogTopK", lambda: run_log_topk(ctx, sc, a, rq, K_SUITE)),
                    ("NearestFeasible",
                     lambda: run_nearest_feasible(ctx, sc, a, rq, K_SUITE) + (None, None))]
            for method, fn in jobs:
                out = fn()
                raw, rt, err = out[0], out[1], out[2]
                extra = out[3:] if len(out) > 3 else (None, None)
                snap = (method == "DiCE")   # retrieval rows are on-grid already
                met, xv = evaluate_candidates(ctx, raw, a, rq, K_SUITE, snap=snap)
                rows.append(dict(dataset=ds, qid=q["qid"], anchor=a["label"],
                                 method=method, k_requested=K_SUITE,
                                 feasible_log_trials=feas_n,
                                 strict_valid_log_trials=(extra[1]
                                     if extra[1] is not None else np.nan),
                                 gen_wall_s=rt, error=err, **met))
                if xv is not None and len(xv):
                    t = xv.copy()
                    t.insert(0, "method", method); t.insert(0, "anchor", a["label"])
                    t.insert(0, "qid", q["qid"]); t.insert(0, "dataset", ds)
                    t["surrogate_pred"] = ctx["predict"](xv)
                    top_rows.append(t)
    R = pd.DataFrame(rows)
    SUITE_RESULTS[ds] = R
    SUITE_TOP[ds] = (pd.concat(top_rows, ignore_index=True)
                     if top_rows else pd.DataFrame())
    display(R.round(4))
    save_table(ctx, R.round(4), f"query_suite_{ds}")
    print(f"[{ds}] suite complete: {len(R)} cells "
          f"({R['error'].notna().sum()} generator errors, reported not hidden).")


100%|██████████| 1/1 [00:00<00:00,  5.52it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  5.60it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  5.67it/s]


...

100%|██████████| 1/1 [00:00<00:00,  6.20it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  6.80it/s]


,dataset,qid,anchor,method,k_requested,feasible_log_trials,strict_valid_log_trials,gen_wall_s,error,raw_n,raw_valid_n,repairs,final_valid_n,shortfall,mean_surrogate_f1,mean_distance
0,proditec,broad_band_free,p50,DiCE,8,2999,NaN,0.1413,None,40,40,35,8,0,0.7903,1.8839
1,proditec,broad_band_free,p50,LogTopK,8,2999,1273.0,0.0011,None,8,8,0,7,1,0.8973,3.8250
2,proditec,broad_band_free,p50,NearestFeasible,8,2999,NaN,0.0024,None,8,8,0,8,0,0.7882,0.3991
3,proditec,broad_band_free,p85,DiCE,8,2999,NaN,0.1322,None,40,40,17,8,0,0.8877,0.3420
4,proditec,broad_band_free,p85,LogTopK,8,2999,1273.0,0.0010,None,8,8,0,7,1,0.8973,0.1793
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,proditec,combo_q,p85,LogTopK,8,267,0.0,0.0010,None,0,0,0,0,8,NaN,NaN
104,proditec,combo_q,p85,NearestFeasible,8,267,NaN,0.0010,None,0,0,0,0,8,NaN,NaN
105,proditec,combo_q,best,DiCE,8,267,NaN,0.1513,None,40,0,12,0,8,NaN,NaN
106,proditec,combo_q,best,LogTopK,8,267,0.0,0.0010,None,0,0,0,0,8,NaN,NaN


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/query_suite_proditec.csv
[proditec] suite complete: 108 cells (7 generator errors, reported not hidden).


## S8 — Generation vs retrieval: where does the generator add value? (R2, item 2)

Aggregates the suite: per query, satisfaction rates and the log-coverage statistic.
Expected pattern (to verify): under free/broad constraints retrieval matches or beats
DiCE; under tight, relative, or combined constraints `strict_valid_log_trials`
shrinks and generation is needed to fill K. Also exports the **confirmation candidates**
for the small retraining pass (notebook-5 patch): top-2 per method for the three
hardest queries.

In [12]:
# ============================================================
# S8) Added-value aggregation + export candidates  (v2)
# - per-dataset selection: absolute confirm queries first, quantile
#   fallback when they yield nothing (MVTec case)
# - never overwrites candidates for a dataset whose confirmation
#   run is already done (protects finished Proditec results)
# - loud diagnostics when selection is empty
# ============================================================
CONFIRM_QIDS_PRIMARY = ["tight_band", "relative_budget_090", "combo_tight"]
CONFIRM_QIDS_FALLBACK = ["tight_band_q", "relative_budget_090", "combo_q"]
CONFIRM_TOP_PER_CELL = 2
CONFIRM_ANCHOR = "p85"

for ds, ctx in CTX.items():
    R = SUITE_RESULTS[ds]
    agg = (R.groupby(["qid", "method"])
             .agg(cells=("anchor", "size"),
                  mean_final_valid=("final_valid_n", "mean"),
                  mean_shortfall=("shortfall", "mean"),
                  mean_raw_valid=("raw_valid_n", "mean"),
                  mean_repairs=("repairs", "mean"),
                  mean_surrogate_f1=("mean_surrogate_f1", "mean"),
                  mean_distance=("mean_distance", "mean"),
                  mean_wall_s=("gen_wall_s", "mean"),
                  mean_feasible_log=("feasible_log_trials", "mean"),
                  mean_strictvalid_log=("strict_valid_log_trials", "mean"))
             .reset_index())
    display(agg.round(3))
    save_table(ctx, agg.round(4), f"query_suite_aggregate_{ds}")

    conf_done = (ctx["out_tables"] / f"baseline_confirmation_{ds}.csv").exists()
    if conf_done:
        print(f"[{ds}] baseline_confirmation_{ds}.csv already exists — keeping the "
              "existing candidate/confirmation pair; NOT overwriting.")
        continue

    T = SUITE_TOP[ds]
    if T.empty:
        print(f"[{ds}] no valid candidates anywhere in the suite."); continue

    def select(qids):
        return (T[(T["qid"].isin(qids)) & (T["anchor"] == CONFIRM_ANCHOR)]
                .sort_values(["qid", "method", "surrogate_pred"],
                             ascending=[True, True, False])
                .groupby(["qid", "method"], as_index=False)
                .head(CONFIRM_TOP_PER_CELL).reset_index(drop=True))

    sel = select(CONFIRM_QIDS_PRIMARY)
    used = "primary(absolute)"
    if sel.empty:
        sel = select(CONFIRM_QIDS_FALLBACK)
        used = "fallback(quantile-based)"
    if sel.empty:
        print(f"[{ds}] STILL EMPTY after quantile fallback — suite diagnostics for "
              "the confirm queries:")
        cols = ["qid", "anchor", "method", "final_valid_n", "shortfall",
                "feasible_log_trials", "strict_valid_log_trials", "error"]
        display(R[R["qid"].isin(CONFIRM_QIDS_PRIMARY + CONFIRM_QIDS_FALLBACK)][cols])
        continue

    sel.insert(0, "confirm_id", range(len(sel)))
    p = ctx["out_tables"] / f"baseline_confirm_candidates_{ds}.csv"
    sel.to_csv(p, index=False)
    print(f"[{ds}] exported {len(sel)} candidates via {used} -> {p}")
    print("      Next: run the appended Section R of notebook 5/5b, then re-run S9.")


,qid,method,cells,mean_final_valid,mean_shortfall,mean_raw_valid,mean_repairs,mean_surrogate_f1,mean_distance,mean_wall_s,mean_feasible_log,mean_strictvalid_log
0,broad_band_free,DiCE,3,8.000,0.000,40.000,19.667,0.796,1.012,0.161,9976.000,NaN
1,broad_band_free,LogTopK,3,8.000,0.000,8.000,0.000,0.815,2.824,0.001,9976.000,2239.000
2,broad_band_free,NearestFeasible,3,3.667,4.333,8.000,0.000,0.789,1.229,0.003,9976.000,NaN
3,combo_q,DiCE,3,0.000,8.000,0.000,4.667,NaN,NaN,0.159,807.000,NaN
4,combo_q,LogTopK,3,0.000,8.000,0.000,0.000,NaN,NaN,0.001,807.000,0.000
5,combo_q,NearestFeasible,3,0.000,8.000,0.000,0.000,NaN,NaN,0.001,807.000,NaN
6,combo_tight,DiCE,3,0.000,8.000,0.000,0.000,NaN,NaN,0.183,807.000,NaN
7,combo_tight,LogTopK,3,0.000,8.000,0.000,0.000,NaN,NaN,0.001,807.000,0.000
8,combo_tight,NearestFeasible,3,0.000,8.000,0.000,0.000,NaN,NaN,0.001,807.000,NaN
9,corruption_none_mild,DiCE,3,8.000,0.000,40.000,18.667,0.794,1.386,0.204,8632.000,NaN


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/query_suite_aggregate_mvtec.csv
[mvtec] exported 6 candidates via fallback(quantile-based) -> /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/baseline_confirm_candidates_mvtec.csv
      Next: run the appended Section R of notebook 5/5b, then re-run S9.


,qid,method,cells,mean_final_valid,mean_shortfall,mean_raw_valid,mean_repairs,mean_surrogate_f1,mean_distance,mean_wall_s,mean_feasible_log,mean_strictvalid_log
0,broad_band_free,DiCE,3,8.000,0.000,40.000,22.667,0.856,0.869,0.135,2999.000,NaN
1,broad_band_free,LogTopK,3,7.000,1.000,8.000,0.000,0.897,1.406,0.001,2999.000,1273.000
2,broad_band_free,NearestFeasible,3,6.333,1.667,8.000,0.000,0.855,0.148,0.002,2999.000,NaN
3,combo_q,DiCE,3,0.000,8.000,0.000,8.333,NaN,NaN,0.156,267.000,NaN
4,combo_q,LogTopK,3,0.000,8.000,0.000,0.000,NaN,NaN,0.001,267.000,0.000
5,combo_q,NearestFeasible,3,0.000,8.000,0.000,0.000,NaN,NaN,0.001,267.000,NaN
6,combo_tight,DiCE,3,0.000,8.000,0.000,6.333,NaN,NaN,0.156,267.000,NaN
7,combo_tight,LogTopK,3,0.000,8.000,0.000,0.000,NaN,NaN,0.001,267.000,0.000
8,combo_tight,NearestFeasible,3,0.000,8.000,0.000,0.000,NaN,NaN,0.001,267.000,NaN
9,corruption_none_mild,DiCE,3,6.667,1.333,28.000,19.667,0.842,1.092,0.145,1709.000,NaN


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/query_suite_aggregate_proditec.csv
[proditec] baseline_confirmation_proditec.csv already exists — keeping the existing candidate/confirmation pair; NOT overwriting.


## S9 — Merge confirmed-baseline results (run after the notebook-5/5b patch)

Skips gracefully if `baseline_confirmation_<ds>.csv` does not exist yet.

In [13]:
# ============================================================
# S9) Confirmed generator-vs-retrieval comparison (second pass)
# ============================================================
for ds, ctx in CTX.items():
    p = ctx["out_tables"] / f"baseline_confirmation_{ds}.csv"
    if not p.exists():
        print(f"[{ds}] S9 pending — {p.name} not found "
              "(run the appended confirmation section of notebook 5/5b first).")
        continue
    conf = pd.read_csv(p)
    summ = (conf.groupby(["qid", "method"])
                .agg(n=("confirm_id", "size"),
                     surrogate_f1=("surrogate_pred", "mean"),
                     confirmed_f1=("f1_test", "mean"),
                     gap=("gap_test_minus_surrogate_pred", "mean"))
                .reset_index())
    display(summ.round(4))
    save_table(ctx, summ.round(4), f"baseline_vs_generator_confirmed_{ds}")


[mvtec] S9 pending — baseline_confirmation_mvtec.csv not found (run the appended confirmation section of notebook 5/5b first).


,qid,method,n,surrogate_f1,confirmed_f1,gap
0,tight_band,DiCE,2,0.8962,0.8760,-0.0202
1,tight_band,LogTopK,2,0.8976,0.8878,-0.0098
2,tight_band,NearestFeasible,2,0.8914,0.8822,-0.0092


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/baseline_vs_generator_confirmed_proditec.csv


## S10 — Grouped / duplicate-aware surrogate split (R2, item 4)

TPE trials are not i.i.d.; a shuffled split can place near-identical configurations in
both partitions. Here we (a) quantify exact-signature duplication and cross-partition
leakage under a reproduction of the original stratified split (notebook-3 recipe:
`zero` stratum + 10 quantile bins on non-zero targets, `random_state=SEED`), and
(b) re-estimate fidelity with **GroupKFold(5)** grouped by the snapped feature-tuple
signature, reporting mean±SD alongside the random-split value. Fast CPU-only CatBoost
retraining; the *operational* surrogate is unchanged.

In [14]:
# ============================================================
# S10) Duplicate leakage + grouped-CV fidelity
# ============================================================
from sklearn.model_selection import train_test_split, GroupKFold
from catboost import Pool

CAT_ITERS = 500   # quick, adequate for a fidelity check

for ds, ctx in CTX.items():
    df = ctx["df"]; F, C = ctx["FEATURES"], ctx["CATEGORICAL"]
    y = df[TARGET].reset_index(drop=True)
    X = df[F].reset_index(drop=True)
    sig = signature_series(ctx, df).reset_index(drop=True)

    n_dup_rows = int(sig.duplicated().sum())
    n_groups = sig.nunique()
    print(f"[{ds}] rows={len(df)}  unique-config groups={n_groups}  "
          f"exact-duplicate rows={n_dup_rows} ({100*n_dup_rows/len(df):.1f}%)")

    # (a) reproduce the original split recipe and measure leakage
    is_zero = (y == 0.0)
    strat = pd.Series(index=y.index, dtype="object")
    strat[is_zero] = "zero"
    nz = y[~is_zero]
    strat.loc[nz.index] = pd.qcut(nz, q=10, duplicates="drop").astype(str)
    Xtr, Xte, ytr, yte, str_tr, str_te, sg_tr, sg_te = train_test_split(
        X, y, strat, sig, test_size=0.20, random_state=SEED,
        shuffle=True, stratify=strat)
    leak = float(sg_te.isin(set(sg_tr)).mean())
    print(f"[{ds}] test rows whose exact config signature also appears in train: "
          f"{100*leak:.1f}%")

    def fit_eval(xtr, ytr_, xte, yte_):
        m = CatBoostRegressor(iterations=CAT_ITERS, random_seed=SEED,
                              verbose=0, allow_writing_files=False)
        m.fit(Pool(xtr, ytr_, cat_features=[F.index(c) for c in C]))
        p = m.predict(Pool(xte, cat_features=[F.index(c) for c in C]))
        return (r2_score(yte_, p),
                float(np.sqrt(mean_squared_error(yte_, p))),
                mean_absolute_error(yte_, p))

    r2_rand, rmse_rand, mae_rand = fit_eval(Xtr, ytr, Xte, yte)

    # (b) grouped 5-fold CV by configuration signature
    gkf = GroupKFold(n_splits=5)
    scores = []
    for tr, te in gkf.split(X, y, groups=sig):
        scores.append(fit_eval(X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te]))
    S = np.array(scores)
    out = pd.DataFrame([
        dict(split="random_stratified_80_20 (paper protocol, reproduced)",
             R2=r2_rand, RMSE=rmse_rand, MAE=mae_rand,
             leakage_pct=100*leak),
        dict(split="GroupKFold(5) by exact config signature (mean)",
             R2=S[:, 0].mean(), RMSE=S[:, 1].mean(), MAE=S[:, 2].mean(),
             leakage_pct=0.0),
        dict(split="GroupKFold(5) (sd)",
             R2=S[:, 0].std(), RMSE=S[:, 1].std(), MAE=S[:, 2].std(),
             leakage_pct=np.nan),
    ])
    display(out.round(4))
    save_table(ctx, out.round(4), f"surrogate_split_check_{ds}")
    print(f"[{ds}] NOTE: quick refit at iterations={CAT_ITERS}; the paper's operational "
          "surrogate (full training) is unchanged. Report both numbers; if grouped R2 "
          "is materially lower, present it as the honest generalization estimate.")


[mvtec] rows=9976  unique-config groups=8476  exact-duplicate rows=1500 (15.0%)
[mvtec] test rows whose exact config signature also appears in train: 19.9%


,split,R2,RMSE,MAE,leakage_pct
0,"random_stratified_80_20 (paper protocol, repro...",0.9953,0.0155,0.0088,19.9399
1,GroupKFold(5) by exact config signature (mean),0.9953,0.0157,0.0089,0.0000
2,GroupKFold(5) (sd),0.0005,0.0010,0.0003,NaN


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/surrogate_split_check_mvtec.csv
[mvtec] NOTE: quick refit at iterations=500; the paper's operational surrogate (full training) is unchanged. Report both numbers; if grouped R2 is materially lower, present it as the honest generalization estimate.
[proditec] rows=2999  unique-config groups=2471  exact-duplicate rows=528 (17.6%)
[proditec] test rows whose exact config signature also appears in train: 26.2%


,split,R2,RMSE,MAE,leakage_pct
0,"random_stratified_80_20 (paper protocol, repro...",0.9327,0.0881,0.0457,26.1667
1,GroupKFold(5) by exact config signature (mean),0.9337,0.0870,0.0452,0.0000
2,GroupKFold(5) (sd),0.0078,0.0058,0.0032,NaN


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/surrogate_split_check_proditec.csv
[proditec] NOTE: quick refit at iterations=500; the paper's operational surrogate (full training) is unchanged. Report both numbers; if grouped R2 is materially lower, present it as the honest generalization estimate.


## S11 — End-to-end latency and revised energy accounting (R2, item 5)

The 2 ms figure is a single surrogate call. Here we assemble the operator-visible
what-if path per method:

`parse (LLM) → anchor scoring (full-log surrogate pass) → generation → rank →
explanation (LLM)`

LLM components are read from `revision_outputs/parser_eval_latency.json` written by
the parser-evaluation notebook; until it exists they appear as NaN with a loud note.
Energy figures are recomputed with these end-to-end times, the one-off HPO-log cost is
reported as an amortization curve, and a retrieval-only path is included for
comparison. Hardware note for the paper: **1× NVIDIA RTX A5500 (power-capped to 0.7),
AMD Threadripper PRO 5995WX, 256 GB RAM** — replaces the bare "300 W" assumption.

In [15]:
# ============================================================
# S11) End-to-end latency + energy (revised)
# ============================================================
PATCHCORE_POWER_W = 300.0   # TODO: replace with nvidia-smi-measured mean for the
                            # power-capped A5500 if available; keep as stated estimate
SURROGATE_POWER_W = 65.0
GRID_KGCO2_PER_KWH = 0.66

for ds, ctx in CTX.items():
    ok = ctx["ok_df"]; R = SUITE_RESULTS.get(ds, pd.DataFrame())
    if ok.empty or "runtime_sec" not in ok.columns:
        print(f"[{ds}] SKIP S11 — need confirmation runtimes."); continue

    retrain_s = float(pd.to_numeric(ok["runtime_sec"], errors="coerce").dropna().mean())

    # surrogate single-query latency (measured now, warm)
    one = ctx["df"].sample(1, random_state=SEED)[ctx["FEATURES"]]
    _ = ctx["predict"](ctx["df"].head(8))
    t0 = time.perf_counter()
    for _i in range(200): _ = ctx["predict"](one)
    sur_ms = (time.perf_counter() - t0) / 200 * 1e3

    # anchor scoring = one full-log surrogate pass (measured)
    t0 = time.perf_counter(); _ = ctx["predict"](ctx["df"]); anchor_s = time.perf_counter() - t0

    # LLM component latencies from parser-eval notebook, if present
    llm = {}
    pj = ctx["root"] / "revision_outputs" / "parser_eval_latency.json"
    if pj.exists():
        llm = json.load(open(pj))
        print(f"[{ds}] LLM latency loaded: {llm}")
    else:
        print(f"[{ds}] NOTE: {pj} not found — run the parser-eval notebook to fill "
              "parse/explain latency; NaN placeholders used meanwhile.")
    parse_s = float(llm.get("parse_mean_s", np.nan))
    explain_s = float(llm.get("explain_mean_s", np.nan))

    # per-method generation time: prefer paper CFs, fall back to suite
    gen = {}
    if not ctx["cfs"].empty and "generation_sec" in ctx["cfs"].columns and        "method" in ctx["cfs"].columns:
        gen = ctx["cfs"].groupby("method")["generation_sec"].mean().to_dict()
    if not R.empty:
        for m, v in R.groupby("method")["gen_wall_s"].mean().items():
            gen.setdefault(m, float(v))

    rows = []
    for m, g in sorted(gen.items()):
        e2e = np.nansum([parse_s, anchor_s, float(g), 0.001, explain_s])
        rows.append(dict(method=m, parse_s=parse_s, anchor_scoring_s=anchor_s,
                         generation_s=float(g), rank_s=0.001, explain_s=explain_s,
                         end_to_end_s=e2e,
                         speedup_vs_retraining=retrain_s / e2e if e2e > 0 else np.nan))
    lat = pd.DataFrame(rows)
    display(lat.round(3))
    save_table(ctx, lat.round(4), f"end_to_end_latency_{ds}")
    print(f"[{ds}] single surrogate call: {sur_ms:.2f} ms  |  PatchCore retrain: "
          f"{retrain_s:.1f} s  — the abstract must quote the end-to-end column, "
          "not the raw call.")

    # energy: per-query, with amortization of the one-off log cost
    n_trials = len(ctx["df"])
    log_gpu_h = retrain_s * n_trials / 3600.0
    e_rows = []
    for _, r in lat.iterrows():
        cpu_s = np.nansum([r["anchor_scoring_s"], r["generation_s"], r["rank_s"]])
        # LLM parse/explain run on GPU in the deployed interface; charged at GPU power
        gpu_s = np.nansum([r["parse_s"], r["explain_s"]])
        e_wh = (cpu_s * SURROGATE_POWER_W + gpu_s * PATCHCORE_POWER_W) / 3600.0
        e_rows.append(dict(method=r["method"],
                           energy_per_query_Wh=e_wh,
                           retrain_energy_per_query_Wh=retrain_s*PATCHCORE_POWER_W/3600,
                           avoided_per_1000_queries_kWh=(retrain_s*PATCHCORE_POWER_W
                                                         - e_wh*3600)/3600/1000*1000/1000,
                           one_off_log_cost_GPU_h=log_gpu_h,
                           queries_to_amortize_log=(log_gpu_h*3600)/retrain_s))
    en = pd.DataFrame(e_rows)
    en["avoided_per_1000_queries_kWh"] = (
        (en["retrain_energy_per_query_Wh"] - en["energy_per_query_Wh"]) * 1000 / 1000)
    en["co2_avoided_per_1000_queries_kg"] = en["avoided_per_1000_queries_kWh"] * GRID_KGCO2_PER_KWH
    display(en.round(3))
    save_table(ctx, en.round(4), f"energy_revised_{ds}")
    print(f"[{ds}] one-off log cost ≈ {log_gpu_h:.1f} GPU-h ({n_trials} trials); "
          f"break-even vs retrain-per-query at ≈ {n_trials} queries — state this "
          "amortization explicitly instead of 'reused indefinitely'.")


[mvtec] NOTE: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/parser_eval_latency.json not found — run the parser-eval notebook to fill parse/explain latency; NaN placeholders used meanwhile.


,method,parse_s,anchor_scoring_s,generation_s,rank_s,explain_s,end_to_end_s,speedup_vs_retraining
0,ACFX,NaN,0.019,99.837,0.001,NaN,99.857,0.308
1,CFNOW,NaN,0.019,5.136,0.001,NaN,5.156,5.965
2,DiCE,NaN,0.019,0.364,0.001,NaN,0.385,79.989
3,LogTopK,NaN,0.019,0.001,0.001,NaN,0.022,1425.938
4,NearestFeasible,NaN,0.019,0.002,0.001,NaN,0.022,1397.592


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/end_to_end_latency_mvtec.csv
[mvtec] single surrogate call: 2.10 ms  |  PatchCore retrain: 30.8 s  — the abstract must quote the end-to-end column, not the raw call.


,method,energy_per_query_Wh,retrain_energy_per_query_Wh,avoided_per_1000_queries_kWh,one_off_log_cost_GPU_h,queries_to_amortize_log,co2_avoided_per_1000_queries_kg
0,ACFX,1.803,2.563,0.760,85.228,9976.0,0.502
1,CFNOW,0.093,2.563,2.470,85.228,9976.0,1.630
2,DiCE,0.007,2.563,2.556,85.228,9976.0,1.687
3,LogTopK,0.000,2.563,2.563,85.228,9976.0,1.691
4,NearestFeasible,0.000,2.563,2.563,85.228,9976.0,1.691


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs/tables/energy_revised_mvtec.csv
[mvtec] one-off log cost ≈ 85.2 GPU-h (9976 trials); break-even vs retrain-per-query at ≈ 9976 queries — state this amortization explicitly instead of 'reused indefinitely'.
[proditec] LLM latency loaded: {'parse_mean_s': 29.638327008206396, 'parse_sd_s': 10.481016021692493, 'parse_mean_s_as_deployed': 22.072013769298792, 'explain_mean_s': 22.866131334006788, 'n_utterances': 40, 'model': 'Qwen2.5-3B-Instruct', 'note': 'parse_mean_s is the revised (v2) front-end; the user study ran on the as-deployed parser — cite that figure in the study section.'}


,method,parse_s,anchor_scoring_s,generation_s,rank_s,explain_s,end_to_end_s,speedup_vs_retraining
0,ACFX,29.638,0.007,111.366,0.001,22.866,163.879,0.289
1,CFNOW,29.638,0.007,1.492,0.001,22.866,54.005,0.877
2,DiCE,29.638,0.007,0.337,0.001,22.866,52.849,0.896
3,LogTopK,29.638,0.007,0.001,0.001,22.866,52.514,0.902
4,NearestFeasible,29.638,0.007,0.002,0.001,22.866,52.515,0.902


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/end_to_end_latency_proditec.csv
[proditec] single surrogate call: 2.06 ms  |  PatchCore retrain: 47.4 s  — the abstract must quote the end-to-end column, not the raw call.


,method,energy_per_query_Wh,retrain_energy_per_query_Wh,avoided_per_1000_queries_kWh,one_off_log_cost_GPU_h,queries_to_amortize_log,co2_avoided_per_1000_queries_kg
0,ACFX,6.386,3.947,-2.439,39.456,2999.0,-1.610
1,CFNOW,4.402,3.947,-0.456,39.456,2999.0,-0.301
2,DiCE,4.382,3.947,-0.435,39.456,2999.0,-0.287
3,LogTopK,4.376,3.947,-0.429,39.456,2999.0,-0.283
4,NearestFeasible,4.376,3.947,-0.429,39.456,2999.0,-0.283


Saved table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/tables/energy_revised_proditec.csv
[proditec] one-off log cost ≈ 39.5 GPU-h (2999 trials); break-even vs retrain-per-query at ≈ 2999 queries — state this amortization explicitly instead of 'reused indefinitely'.


## S12 — Output manifest

In [16]:
# ============================================================
# S12) Manifest of everything written by this notebook
# ============================================================
for ds, ctx in CTX.items():
    files = sorted(list(ctx["out_tables"].glob("*")) + list(ctx["out_figures"].glob("*")))
    man = pd.DataFrame({"file": [str(f) for f in files],
                        "kind": ["table" if f.parent.name == "tables" else "figure"
                                 for f in files]})
    man.to_csv(ctx["out_tables"] / "MANIFEST.csv", index=False)
    print(f"[{ds}] {len(man)} outputs  ->  {ctx['out_tables'].parent}")
    display(man.tail(15))
print("\nDone. Second pass (S9) becomes available after running the appended "
      "confirmation section of notebook 5/5b.")


[mvtec] 34 outputs  ->  /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/revision_outputs


,file,kind
19,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
20,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
21,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
22,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
23,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
24,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
25,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
26,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
27,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table
28,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_m...,table


[proditec] 34 outputs  ->  /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs


,file,kind
19,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
20,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
21,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
22,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
23,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
24,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
25,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
26,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
27,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table
28,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_p...,table



Done. Second pass (S9) becomes available after running the appended confirmation section of notebook 5/5b.
